In [1]:
import pandas as pd
application_df = pd.read_csv('../IS453 Group Assignment - Data/IS453 Group Assignment - Application Data.csv')
bureau_df = pd.read_csv('../IS453 Group Assignment - Data/IS453 Group Assignment - Bureau Data.csv')

# Week 7 Preparation
Unpack the file "IS453 Group Assignment - Data.zip"<BR>
Based on the data files and data dictionary, answer the questions below.

**DIY Q4: Explore Assignment Data**
- How many rows and column in each data file?
- Through which variable is the data in two files linked?
- Is the relationship between the records in the files one-to-one or one-to-many? In which direction?
- Are all of the records in each file linked to the other? If not, which file has unlinked records?
- Which variables would potentially conflict with fair lending principles?

**Answers**
- **Size:** Application 307,511 rows x 120 columns; Bureau 1,716,428 rows x 17 columns.
- **Link:** `SK_ID_CURR`.
- **Relationship:** one-to-many, from Application to Bureau. `SK_ID_CURR` is unique in Application, while Bureau has 1 to 116 rows per ID (mean 5.6, median 4). `SK_ID_BUREAU` is unique per Bureau row.
- **Unlinked records:** Not all records link. 263,491 IDs appear in both files. 44,020 application IDs have no bureau history (these need a "no bureau record" flag/treatment after the merge). 42,320 bureau IDs have no application row, so these are unusable for modelling.
- **Fair lending concerns:** `CODE_GENDER`, `DAYS_BIRTH` (age), `NAME_FAMILY_STATUS`, `CNT_CHILDREN`, `CNT_FAM_MEMBERS` (family status/dependants), `NAME_EDUCATION_TYPE` (possible proxy), `REGION_POPULATION_RELATIVE`, `REGION_RATING_CLIENT(_W_CITY)` and the housing-block variables (geographic proxies / redlining risk), `ORGANIZATION_TYPE` and `NAME_INCOME_TYPE` (e.g. maternity leave), `NAME_TYPE_SUITE`, and the `*_CNT_SOCIAL_CIRCLE` variables (judged on others' behaviour, not the applicant's).

In [2]:
print('Application:', application_df.shape, '| Bureau:', bureau_df.shape)

app_ids, bur_ids = set(application_df.SK_ID_CURR), set(bureau_df.SK_ID_CURR)
print('Application IDs with no bureau record:', len(app_ids - bur_ids))
print('Bureau IDs with no application record:', len(bur_ids - app_ids))
print('Overall bad rate:', application_df.TARGET.mean()*100)

Application: (307511, 120) | Bureau: (1716428, 17)
Application IDs with no bureau record: 44020
Bureau IDs with no application record: 42320
Overall bad rate: 8.072881945686495


# Week 9 Preparation
Based on the group assignment data, answer the DIY question below.

**DIY Q4: Preparing the bureau data for merging**

To combine the bureau data with the application data you will need to "flatten" the multiple rows associated with each application ID (SK_ID_CURR) into a single row. <BR>
It will also be helpful if you can reduce the number of columns in the bureau data that will need to be merged.
- Based on the data dictionary definitions and the inspection of the bureau data, explain how you would propose to do feature engineering on all of the bureau variables so that they can be condensed to one value per application ID.
- Calculate the correlation between the bureau variables and use that information to further propose how columns could potentially be combined or dropped.

Put your code and answers below, and prepare to discuss in class next week.

In [3]:
import numpy as np

# Correlation on the raw bureau numeric columns
raw_corr = bureau_df.drop(columns=['SK_ID_CURR', 'SK_ID_BUREAU']).select_dtypes('number').corr()
print(raw_corr.round(2))

b = bureau_df.copy()
biz = ['Loan for business development', 'Loan for working capital replenishment', 'Loan for the purchase of equipment']
b['ACTIVE'] = (b.CREDIT_ACTIVE == 'Active').astype(int)
b['CLOSED'] = (b.CREDIT_ACTIVE == 'Closed').astype(int)
b['SOLD_BAD'] = b.CREDIT_ACTIVE.isin(['Sold', 'Bad debt']).astype(int)
b['CARD'] = (b.CREDIT_TYPE == 'Credit card').astype(int)
b['CONSUMER'] = (b.CREDIT_TYPE == 'Consumer credit').astype(int)
b['MORTGAGE'] = (b.CREDIT_TYPE == 'Mortgage').astype(int)
b['CAR'] = (b.CREDIT_TYPE == 'Car loan').astype(int)
b['MICRO'] = (b.CREDIT_TYPE == 'Microloan').astype(int)
b['BUSINESS'] = b.CREDIT_TYPE.isin(biz).astype(int)
b['OVERDUE_NOW'] = (b.CREDIT_DAY_OVERDUE > 0).astype(int)

bureau_flat = b.groupby('SK_ID_CURR').agg(
    BUR_N=('SK_ID_BUREAU', 'count'),
    BUR_N_ACTIVE=('ACTIVE', 'sum'), BUR_N_CLOSED=('CLOSED', 'sum'), BUR_N_SOLD_BAD=('SOLD_BAD', 'sum'),
    BUR_N_CARD=('CARD', 'sum'), BUR_N_CONSUMER=('CONSUMER', 'sum'), BUR_N_MORTGAGE=('MORTGAGE', 'sum'),
    BUR_N_CAR=('CAR', 'sum'), BUR_N_MICRO=('MICRO', 'sum'), BUR_N_BUSINESS=('BUSINESS', 'sum'),
    BUR_N_OVERDUE_NOW=('OVERDUE_NOW', 'sum'), BUR_MAX_DAYS_OVERDUE=('CREDIT_DAY_OVERDUE', 'max'),
    BUR_MAX_AMT_MAX_OVERDUE=('AMT_CREDIT_MAX_OVERDUE', 'max'), BUR_SUM_AMT_OVERDUE=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    BUR_SUM_PROLONG=('CNT_CREDIT_PROLONG', 'sum'),
    BUR_MOST_RECENT_DAYS=('DAYS_CREDIT', 'max'), BUR_OLDEST_DAYS=('DAYS_CREDIT', 'min'),
    BUR_MAX_ENDDATE=('DAYS_CREDIT_ENDDATE', 'max'), BUR_LAST_UPDATE=('DAYS_CREDIT_UPDATE', 'max'),
    BUR_SUM_CREDIT=('AMT_CREDIT_SUM', 'sum'), BUR_SUM_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUR_SUM_LIMIT=('AMT_CREDIT_SUM_LIMIT', 'sum'), BUR_SUM_ANNUITY=('AMT_ANNUITY', 'sum'),
).reset_index()
bureau_flat['BUR_DEBT_RATIO'] = bureau_flat.BUR_SUM_DEBT / bureau_flat.BUR_SUM_CREDIT.replace(0, np.nan)
bureau_flat['BUR_PCT_ACTIVE'] = bureau_flat.BUR_N_ACTIVE / bureau_flat.BUR_N
print(bureau_flat.shape, 'unique IDs:', bureau_flat.SK_ID_CURR.is_unique)

# Correlation on the flattened features: list highly correlated pairs
c = bureau_flat.drop(columns='SK_ID_CURR').corr()
pairs = c.where(np.triu(np.ones(c.shape), 1).astype(bool)).stack()
print(pairs[pairs.abs() > 0.8].sort_values(ascending=False).round(2))

                        DAYS_CREDIT  CREDIT_DAY_OVERDUE  DAYS_CREDIT_ENDDATE  \
DAYS_CREDIT                    1.00               -0.03                 0.23   
CREDIT_DAY_OVERDUE            -0.03                1.00                -0.01   
DAYS_CREDIT_ENDDATE            0.23               -0.01                 1.00   
DAYS_ENDDATE_FACT              0.88               -0.01                 0.25   
AMT_CREDIT_MAX_OVERDUE        -0.01                0.00                 0.00   
CNT_CREDIT_PROLONG            -0.03                0.00                 0.11   
AMT_CREDIT_SUM                 0.05               -0.00                 0.06   
AMT_CREDIT_SUM_DEBT            0.14               -0.00                 0.08   
AMT_CREDIT_SUM_LIMIT           0.03               -0.00                 0.10   
AMT_CREDIT_SUM_OVERDUE        -0.00                0.09                 0.00   
DAYS_CREDIT_UPDATE             0.69               -0.02                 0.25   
AMT_ANNUITY                    0.01     

**Proposed feature engineering (one row per `SK_ID_CURR`)**

| Bureau variable | Aggregation(s) | Rationale |
|---|---|---|
| `SK_ID_BUREAU` | count -> `BUR_N` | Number of prior credit lines (credit experience / activity) |
| `CREDIT_ACTIVE` | count Active / Closed / Sold+Bad debt | Sold and Bad debt are tiny (6.5k and 21 rows), so they are merged into one count. Active share shows current exposure |
| `CREDIT_CURRENCY` | **drop** | 99.9% is "currency 1", so no information |
| `CREDIT_TYPE` | count of Card, Consumer, Mortgage, Car, Microloan, **Business** (business development + working capital + equipment) | 15 types are too many and most are rare. Business credit history is directly relevant to our small-business segment |
| `DAYS_CREDIT` | max (most recent application), min (oldest = history length) | Credit recency and length of history |
| `CREDIT_DAY_OVERDUE` | max, count of lines currently overdue | Current delinquency is the strongest risk signal |
| `AMT_CREDIT_MAX_OVERDUE` | max | Worst past delinquency (65% missing; treat missing as "no overdue recorded" and consider a flag) |
| `AMT_CREDIT_SUM_OVERDUE` | sum | Total amount currently overdue |
| `CNT_CREDIT_PROLONG` | sum | Number of extensions = possible repayment stress |
| `DAYS_CREDIT_ENDDATE` | max | Longest remaining commitment |
| `DAYS_ENDDATE_FACT` | **drop** | Only exists for closed credits, and correlated 0.88 with `DAYS_CREDIT` |
| `AMT_CREDIT_SUM` / `_DEBT` / `_LIMIT` | sum; plus ratio `DEBT/CREDIT` | Total exposure and leverage. A ratio is more comparable across applicants than raw amounts |
| `AMT_ANNUITY` | sum | Instalment burden (72% missing, so interpret with care) |
| `DAYS_CREDIT_UPDATE` | max | Freshness of bureau information |

Applicants with no bureau record (44,020) will get counts = 0 and a `NO_BUREAU` flag after the left join (to add at merge time), with other aggregates left missing for binning. This avoids confusing "no history" with "clean history".

**Correlation findings**
- On the raw data: `DAYS_CREDIT` vs `DAYS_ENDDATE_FACT` is 0.88 and `DAYS_CREDIT` vs `DAYS_CREDIT_UPDATE` is 0.69, which supports dropping `DAYS_ENDDATE_FACT`. `AMT_CREDIT_SUM` vs `AMT_CREDIT_SUM_DEBT` is 0.68, which supports using the debt ratio.
- After aggregation, run the pairs check in the code cell and drop one of any pair with |r| > 0.8. In my run, `BUR_N` vs `BUR_N_CONSUMER` (0.93) and `BUR_N` vs `BUR_N_CLOSED` (0.93) came out high. Keep `BUR_N` and use the type/status mix as shares, or drop the redundant counts.

# Week 10 Consult with Prof
**Our customer segment is: Launch small business loan**

In Week 10, each group will explain to the professor their criteria for selecting rows from the data that are representative of their customer segment and the resulting:
- number of rows
- percentage of bads in comparison to the original data set

Each group will also describe its intended approach for merging the application and bureau data sets.


In [4]:
application_df = application_df.copy()     # stops pandas showing a "DataFrame is highly fragmented" warning
application_df['AGE'] = -application_df.DAYS_BIRTH / 365
segment_df = application_df[application_df.AGE.between(18, 55) & (application_df.AMT_CREDIT < 500000)].copy()
print('Rows:', len(segment_df), f'({len(segment_df) / len(application_df):.1%} of original)')
print('Bad rate: segment', f'{segment_df.TARGET.mean():.2%}', '| original', f'{application_df.TARGET.mean():.2%}')

# Left-join the flattened bureau table (from the Week 9 cell) onto the segment
merged_df = segment_df.merge(bureau_flat, on='SK_ID_CURR', how='left')
merged_df['NO_BUREAU'] = merged_df.BUR_N.isna().astype(int)
count_cols = [c for c in bureau_flat.columns if c.startswith('BUR_N')]
merged_df[count_cols] = merged_df[count_cols].fillna(0)

print('Merged shape:', merged_df.shape, '| unique IDs:', merged_df.SK_ID_CURR.is_unique)
print('No bureau record:', merged_df.NO_BUREAU.sum(), f'({merged_df.NO_BUREAU.mean():.1%})')
print(merged_df.groupby('NO_BUREAU').TARGET.agg(['size', 'mean']))

Rows: 115022 (37.4% of original)
Bad rate: segment 9.49% | original 8.07%
Merged shape: (115022, 147) | unique IDs: True
No bureau record: 18476 (16.1%)
            size      mean
NO_BUREAU                 
0          96546  0.090610
1          18476  0.117558


**Checkpoint summary: Launch small business loan**

*Row selection criteria (both approved by the professor)*
1. **Age 18 to 55**: the realistic window for starting a business. The youngest applicant is 20.5, so the lower bound removes nobody. The 55 cap removes about 22% of rows.
2. **`AMT_CREDIT` < 500,000**: a plausible startup financing range (equipment, inventory, deposits, initial operating expenses). 500k is about the median loan, so it keeps the smaller half. It leaves 115k rows and about 11k bads, enough to model, whereas lower caps shrink the sample sharply (e.g. 150k leaves 909 bads) and higher caps add loans that look less like startup financing.

*Result*
- **Rows:** 115,022 of 307,511 (37.4% of the original data)
- **Bads:** 10,920 = **9.49%**, against 8.07% in the original data (the segment is somewhat riskier than the full book)

*Merge approach*
1. Filter the application data first.
2. Flatten the bureau data to one row per `SK_ID_CURR` (Week 9 aggregations).
3. Left-join the flattened bureau table onto the filtered applications on `SK_ID_CURR`. This keeps all 115,022 applicants. The result has one row per applicant (confirmed unique).
4. 18,476 applicants (16.1%) have no bureau record. For them the count features are set to 0, the other aggregates stay missing, and a `NO_BUREAU` flag is added. These applicants have a higher bad rate (11.8% vs 9.1% with bureau history), so the flag is informative.

*Assumptions and limitations*
- No variable records the loan purpose, so age and loan size are proxies. Not every applicant in the segment is actually starting a business.
- About 19% of the filtered loans are revolving rather than term loans.
- The amount cap biases the segment towards smaller borrowers.

In [5]:
age_ok = application_df[application_df.AGE.between(18, 55)]
base_rows = (age_ok.AMT_CREDIT < 500000).sum()

rows = []
for cap in [500000, 450000, 400000, 350000, 300000, 250000, 200000, 150000, 100000]:
    sub = age_ok[age_ok.AMT_CREDIT < cap]
    rows.append({'AMT_CREDIT <': f'{cap:,}', 'Rows': len(sub), '% of 500k sample': round(len(sub) / base_rows * 100, 1),
                 'Bads': int(sub.TARGET.sum()), 'Bad rate %': round(sub.TARGET.mean() * 100, 2)})
cap_table = pd.DataFrame(rows)
print(cap_table.to_string(index=False))

AMT_CREDIT <   Rows  % of 500k sample  Bads  Bad rate %
     500,000 115022             100.0 10920        9.49
     450,000  95803              83.3  8946        9.34
     400,000  89050              77.4  8113        9.11
     350,000  81053              70.5  7252        8.95
     300,000  68103              59.2  5839        8.57
     250,000  44331              38.5  3628        8.18
     200,000  27607              24.0  2159        7.82
     150,000  13128              11.4   909        6.92
     100,000   4185               3.6   266        6.36


# Week 10 Preparation

**Do fine classing and characteristic screening on your project data**

Use the functions in the earlier part of the lab to calculate WOE and IV on a subset of your poject data.

**DIY Q4: Calculate IV for a subset of your variables**

Based on correlation to the target variable, pick the 30 characteristics with the highest correlation and calculate their IV.

In [6]:
def woe_iv(data):
    working_data = data.copy()
    working_data['Bin_Range'] = working_data['Bin_Range'].astype('object')
    working_data['Bin_Range'] = working_data['Bin_Range'].fillna('Missing')
    variable_data = pd.DataFrame()
    variable_data['Bin_Range'] = working_data.groupby(by='Bin_Range', as_index=False).count()['Bin_Range']
    variable_data['Count'] = working_data.groupby(by='Bin_Range', as_index=False).count()['BAD']
    variable_data['Events'] = working_data.groupby(by='Bin_Range', as_index=False).sum()['BAD']
    variable_data['Non_Events'] = variable_data['Count'] - variable_data['Events']
    variable_data['%_of_Events'] = variable_data['Events']/sum(variable_data['Events'])
    variable_data['%_of_Non_Events'] = variable_data['Non_Events']/sum(variable_data['Non_Events'])
    variable_data['WOE'] = np.log(variable_data['%_of_Non_Events'] / variable_data['%_of_Events'])
    variable_data['IV'] = (variable_data['%_of_Non_Events']-variable_data['%_of_Events']) * variable_data['WOE']
    variable_data['total_IV'] = variable_data['IV'].sum()
    return(variable_data)

screen_df = merged_df.drop(columns=['SK_ID_CURR', 'DAYS_BIRTH'])
corr_matrix = screen_df.select_dtypes('number').corr()
ranked = corr_matrix['TARGET'].abs().sort_values(ascending=False)
top_30 = ranked.iloc[1:31].reset_index()

screen_df['DAYS_LAST_PHONE_CHANGE'] = screen_df['DAYS_LAST_PHONE_CHANGE'].fillna(0)

def predictive_power(iv):
    if iv < 0.02:
        return 'Not useful'
    if iv < 0.1:
        return 'Weak'
    if iv < 0.3:
        return 'Medium'
    if iv < 0.5:
        return 'Strong'
    return 'Suspicious'

bin_count = 20
rows = []
for variable in top_30['index']:
    woe_bin_data = screen_df[[variable]].copy()
    woe_bin_data['BAD'] = screen_df['TARGET']

    if woe_bin_data.groupby(variable).count().shape[0] > bin_count:
        woe_bin_data['Bin_Range'] = pd.qcut(woe_bin_data[variable], q=bin_count, duplicates='drop')
    else:
        woe_bin_data['Bin_Range'] = pd.cut(woe_bin_data[variable], bins=[-1, 0, 1, 2, 3, 100])

    iv = woe_iv(woe_bin_data).loc[0, 'total_IV']
    rows.append({'Variable': variable, 'Correlation': round(corr_matrix.loc[variable, 'TARGET'], 4),
                 'IV': round(iv, 4), 'Predictive_Power': predictive_power(iv)})

iv_table = pd.DataFrame(rows).sort_values('IV', ascending=False).reset_index().drop(columns=['index'])
display(iv_table)

,Variable,Correlation,IV,Predictive_Power
0,EXT_SOURCE_1,-0.1530,0.1720,Medium
1,BUR_DEBT_RATIO,0.0961,0.0992,Weak
2,BUR_PCT_ACTIVE,0.0822,0.0895,Weak
3,BUR_OLDEST_DAYS,0.0838,0.0855,Weak
4,BUR_MOST_RECENT_DAYS,0.0450,0.0836,Weak
5,DAYS_LAST_PHONE_CHANGE,0.0667,0.0611,Weak
6,REGION_RATING_CLIENT_W_CITY,0.0645,0.0485,Weak
7,BUR_N_CLOSED,-0.0482,0.0471,Weak
8,AGE,-0.0596,0.0462,Weak
9,REGION_RATING_CLIENT,0.0612,0.0441,Weak


# Credit scorecard model (Team 6)

The cells below extend the code from previously. The model uses a stratified train/test split, learns fine-classing cut points from the training data only, calculates WOE and IV, fits logistic regression on WOE values, and converts the model into a points scorecard. `DAYS_BIRTH` and the explicitly identified fairness-sensitive fields are excluded.

## Choice of Final Variables

The final variables were selected using four main criteria. First, they had to be relevant to repayment risk and the small-business loan decision, such as affordability, employment stability, external risk information, or previous credit behaviour. Second, variables were screened using Fine IV and Coarse IV calculated from the training data; variables were retained only when their final Coarse IV was at least 0.02. Third, highly correlated variables were reduced so that the scorecard did not include several variables measuring essentially the same characteristic. Finally, the variables had to satisfy fair-lending considerations, have final bins containing at least 5% of training applicants, and show a monotonic or categorical WOE pattern.

Non-monotonic candidates were reviewed using their actual coarse WOE sequences. They were removed when the sequence did not show a clear, defensible business shape such as a stable threshold or U-shaped relationship. The preliminary IV analysis was used only as exploratory guidance; formal selection used training-only Fine IV and Coarse IV screening. The accepted variables were WOE-encoded and used as inputs to the logistic regression scorecard.


## 1. Set up the modelling data and select preliminary features (which undergo further screening), split into train and test data


In [7]:
# Import the course modelling tools used for splitting, screening, regression, and evaluation.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Build the initial candidate pool from all available numeric risk variables.
# The preliminary IV table remains exploratory; it does not gate this list.
fair_lending_exclusions = {
    'CODE_GENDER', 'NAME_FAMILY_STATUS', 'NAME_EDUCATION_TYPE',
    'NAME_HOUSING_TYPE', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'DAYS_BIRTH', 'REGION_POPULATION_RELATIVE', 'REGION_RATING_CLIENT',
    'REGION_RATING_CLIENT_W_CITY', 'REG_REGION_NOT_LIVE_REGION',
    'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION',
    'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY',
    'LIVE_CITY_NOT_WORK_CITY', 'OBS_30_CNT_SOCIAL_CIRCLE',
    'DEF_30_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE',
    'DEF_60_CNT_SOCIAL_CIRCLE'
}
identifier_or_target = {'SK_ID_CURR', 'TARGET', 'AGE'}
excluded_columns = fair_lending_exclusions | identifier_or_target
model_features = [
    column for column in merged_df.select_dtypes(include='number').columns
    if column not in excluded_columns
]

# Create the modelling table and clean invalid numeric values.
model_df = merged_df[model_features + ['TARGET']].copy()
model_df = model_df.replace([np.inf, -np.inf], np.nan)
if 'DAYS_EMPLOYED' in model_df:
    model_df['DAYS_EMPLOYED'] = model_df['DAYS_EMPLOYED'].replace(365243, np.nan)
model_df['TARGET'] = model_df['TARGET'].astype(int)

# Split first so every later binning, IV, correlation, and WOE decision uses training data only.
train_df, test_df = train_test_split(
    model_df, test_size=0.30, random_state=453, stratify=model_df['TARGET']
)
print('Candidate variables:', model_features)
print('Train:', train_df.shape, '| Test:', test_df.shape)
print('Train bad rate:', f"{train_df.TARGET.mean():.2%}", '| Test bad rate:', f"{test_df.TARGET.mean():.2%}")


Candidate variables: ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'HOUR_APPR_PROCESS_START', 'EXT_SOURCE_1', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_

## 2. Create functions for fine classing, coarse classing and calculating WOE and IV


In [8]:
# Fine classing uses equal-frequency bins for continuous variables.
def make_fine_model(series, max_bins=10):
    non_missing = series.dropna()
    unique_values = np.sort(non_missing.unique())

    # Low-cardinality count/binary variables retain their observed categories.
    if len(unique_values) <= max_bins:
        return {'kind': 'categorical', 'labels': [str(v) for v in unique_values]}

    # For continuous variables, learn quantile cut points from training data only.
    _, edges = pd.qcut(non_missing, q=max_bins, retbins=True, duplicates='drop')
    edges = np.asarray(edges, dtype=float)
    edges[0], edges[-1] = -np.inf, np.inf
    return {'kind': 'numeric', 'edges': edges}


def assign_fine_bins(series, fine_model):
    if fine_model['kind'] == 'categorical':
        return series.map(lambda value: str(value) if pd.notna(value) else 'Missing').astype(str)
    labels = pd.cut(series, bins=fine_model['edges'], include_lowest=True)
    return labels.astype(object).where(series.notna(), 'Missing').astype(str)


def woe_table_from_bins(binned, target):
    table = pd.DataFrame({'bin': binned, 'bad': target}).groupby('bin', sort=False)['bad'].agg(
        count='size', bad='sum'
    )
    table['good'] = table['count'] - table['bad']
    total_bad, total_good = table['bad'].sum(), table['good'].sum()
    table['dist_bad'] = (table['bad'] + 0.5) / (total_bad + 0.5 * len(table))
    table['dist_good'] = (table['good'] + 0.5) / (total_good + 0.5 * len(table))
    table['woe'] = np.log(table['dist_good'] / table['dist_bad'])
    table['iv'] = (table['dist_good'] - table['dist_bad']) * table['woe']
    return table


def iv_strength(iv):
    if iv < 0.02: return 'Not useful'
    if iv < 0.10: return 'Weak'
    if iv < 0.30: return 'Medium'
    if iv < 0.50: return 'Strong'
    return 'Suspicious'


def add_group_woe(groups):
    total_bad = sum(g['bad'] for g in groups)
    total_good = sum(g['count'] - g['bad'] for g in groups)
    for g in groups:
        g['good'] = g['count'] - g['bad']
        g['dist_bad'] = (g['bad'] + 0.5) / (total_bad + 0.5 * len(groups))
        g['dist_good'] = (g['good'] + 0.5) / (total_good + 0.5 * len(groups))
        g['woe'] = np.log(g['dist_good'] / g['dist_bad'])
    return groups


def coarse_class(train_bins, target, fine_model, min_bin_pct=0.05, woe_tolerance=0.05):
    """Create a 5%-minimum coarse solution and avoid unnecessary IV loss."""
    fine_table = woe_table_from_bins(train_bins, target)
    if fine_model['kind'] == 'categorical':
        mapping = {label: label for label in fine_table.index}
        return mapping, fine_table, fine_table.copy(), 'Categorical'

    labels = [label for label in fine_table.index if label != 'Missing']
    groups = [
        {'fine_labels': [label], 'count': int(fine_table.loc[label, 'count']),
         'bad': int(fine_table.loc[label, 'bad'])}
        for label in labels
    ]
    if 'Missing' in fine_table.index:
        groups.append({'fine_labels': ['Missing'],
                       'count': int(fine_table.loc['Missing', 'count']),
                       'bad': int(fine_table.loc['Missing', 'bad'])})

    min_count = int(np.ceil(min_bin_pct * len(train_bins)))
    add_group_woe(groups)

    def merge(left, right):
        return {'fine_labels': left['fine_labels'] + right['fine_labels'],
                'count': left['count'] + right['count'],
                'bad': left['bad'] + right['bad']}

    # Enforce the minimum size first. Missing is allowed to merge with its
    # closest numeric neighbour, but is never used to determine trend.
    while len(groups) > 1 and any(group['count'] < min_count for group in groups):
        small_index = next(i for i, group in enumerate(groups) if group['count'] < min_count)
        neighbours = [i for i in (small_index - 1, small_index + 1)
                      if 0 <= i < len(groups)]
        neighbour_index = min(neighbours,
                              key=lambda i: abs(groups[small_index]['woe'] - groups[i]['woe']))
        low, high = sorted([small_index, neighbour_index])
        groups[low] = merge(groups[low], groups[high])
        del groups[high]
        add_group_woe(groups)

    size_groups = groups

    def direction_and_stability(candidate):
        numeric = [g for g in candidate if 'Missing' not in g['fine_labels']]
        differences = np.diff([g['woe'] for g in numeric])
        meaningful = [d for d in differences if abs(d) > woe_tolerance]
        if not meaningful:
            return 'Stable non-monotonic', 0
        direction = 1 if sum(d > 0 for d in meaningful) >= sum(d < 0 for d in meaningful) else -1
        violations = sum((d < -woe_tolerance if direction == 1 else d > woe_tolerance)
                         for d in differences)
        return direction, violations

    def monotonic_candidate(candidate):
        candidate = [dict(g, fine_labels=list(g['fine_labels'])) for g in candidate]
        numeric = [g for g in candidate if 'Missing' not in g['fine_labels']]
        missing = [g for g in candidate if 'Missing' in g['fine_labels']]
        add_group_woe(numeric)
        direction, _ = direction_and_stability(numeric)
        increasing = direction != -1
        while len(numeric) > 1:
            violation = next((i for i in range(len(numeric) - 1)
                              if ((increasing and numeric[i + 1]['woe'] < numeric[i]['woe'] - woe_tolerance)
                                  or (not increasing and numeric[i + 1]['woe'] > numeric[i]['woe'] + woe_tolerance))), None)
            if violation is None:
                break
            numeric[violation] = merge(numeric[violation], numeric[violation + 1])
            del numeric[violation + 1]
            add_group_woe(numeric)
        return numeric + missing

    monotonic_groups = monotonic_candidate(size_groups)

    def groups_to_table(candidate):
        add_group_woe(candidate)
        rows = []
        mapping = {}
        for number, group in enumerate(candidate, start=1):
            label = f'Coarse bin {number}'
            for fine_label in group['fine_labels']:
                mapping[fine_label] = label
            rows.append({'Bin': label, 'Count': group['count'], 'Bad': group['bad'],
                         'Bad_Rate': group['bad'] / group['count'], 'WOE': group['woe'],
                         'IV': (group['dist_good'] - group['dist_bad']) * group['woe']})
        return mapping, pd.DataFrame(rows).set_index('Bin')

    size_mapping, size_table = groups_to_table(size_groups)
    monotonic_mapping, monotonic_table = groups_to_table(monotonic_groups)
    size_iv = size_table['IV'].sum()
    monotonic_iv = monotonic_table['IV'].sum()
    _, violations = direction_and_stability(size_groups)
    stable_nonmonotonic = violations <= 2

    # Prefer monotonic WOE for a conventional scorecard when it remains useful.
    # If monotonic merging destroys too much IV, retain the size-compliant
    # alternative only when it remains above the final IV threshold and has
    # limited violations; otherwise the caller drops the variable.
    if monotonic_iv >= 0.02:
        mapping, coarse_table = monotonic_mapping, monotonic_table
        trend = 'Monotonic' if len(monotonic_groups) == len(size_groups) else 'Monotonic after material merges'
    elif stable_nonmonotonic and size_iv >= 0.02:
        mapping, coarse_table = size_mapping, size_table
        trend = 'Stable non-monotonic'
    else:
        mapping, coarse_table = monotonic_mapping, monotonic_table
        trend = 'Monotonic below IV threshold'
    return mapping, fine_table, coarse_table, trend


## 3. Perform fine classing, calculate WOE and IV for fine bins, then perform coarse classing

Fine bins are learned from the training sample only. Fine IV is used for formal screening, followed by training-data correlation pruning.

For coarse classing, adjacent fine bins are merged where needed so that every final bin contains at least 5% of training applicants. The final scorecard prefers monotonic WOE trends; categorical variables are also permitted. Non-monotonic candidates are reviewed using their actual WOE sequences and retained only when a clear, defensible business shape can be stated. In this dataset, the reviewed non-monotonic patterns were too irregular to justify, so they are excluded.

Coarse IV is recalculated from the final bins actually used for WOE encoding. Variables are retained only when Coarse IV is at least 0.02. Fine and Coarse IV strength bands are displayed for review, and any change is reported rather than hidden. The resulting monotonic/categorical survivors form the final scorecard variable set.


In [9]:
# Fine-bin IV screening on every candidate variable, using training data only.
fine_models, fine_bins_train, fine_bins_test = {}, {}, {}
fine_tables, fine_iv_rows = {}, []
for feature in model_features:
    fine_model = make_fine_model(train_df[feature], max_bins=10)
    train_bins = assign_fine_bins(train_df[feature], fine_model)
    test_bins = assign_fine_bins(test_df[feature], fine_model)
    table = woe_table_from_bins(train_bins, train_df['TARGET'])
    fine_models[feature] = fine_model
    fine_bins_train[feature], fine_bins_test[feature] = train_bins, test_bins
    fine_tables[feature] = table
    fine_iv_rows.append({'Variable': feature, 'Fine_IV': table['iv'].sum(), 'Fine_Bins': len(table), 'IV_Strength': iv_strength(table['iv'].sum())})
fine_iv_table = pd.DataFrame(fine_iv_rows).sort_values('Fine_IV', ascending=False).reset_index(drop=True)
display(fine_iv_table.style.format({'Fine_IV': '{:.4f}'}))

# Keep variables with at least weak IV, then remove highly correlated survivors.
iv_survivors = fine_iv_table.loc[fine_iv_table['Fine_IV'] >= 0.02, 'Variable'].tolist()
iv_lookup = fine_iv_table.set_index('Variable')['Fine_IV'].to_dict()
train_corr = train_df[iv_survivors].corr().abs()
final_candidates, correlation_drops = iv_survivors.copy(), []
for i, left in enumerate(iv_survivors):
    for right in iv_survivors[i+1:]:
        if left in final_candidates and right in final_candidates and train_corr.loc[left, right] > 0.80:
            drop = left if iv_lookup[left] < iv_lookup[right] else right
            final_candidates.remove(drop)
            correlation_drops.append({'Dropped': drop, 'Kept': right if drop == left else left, 'Absolute_Correlation': train_corr.loc[left, right]})
print('IV survivors:', iv_survivors)
print('Final candidates after correlation pruning:', final_candidates)
if correlation_drops: display(pd.DataFrame(correlation_drops))


,Variable,Fine_IV,Fine_Bins,IV_Strength
0,EXT_SOURCE_1,0.1686,11,Medium
1,BUR_DEBT_RATIO,0.0942,10,Weak
2,BUR_PCT_ACTIVE,0.0832,9,Weak
3,DAYS_EMPLOYED,0.0818,11,Weak
4,BUR_OLDEST_DAYS,0.0775,11,Weak
5,BUR_MOST_RECENT_DAYS,0.0741,11,Weak
6,DAYS_LAST_PHONE_CHANGE,0.0613,9,Weak
7,BUR_N_CLOSED,0.0457,6,Weak
8,AMT_GOODS_PRICE,0.0424,11,Weak
9,BUR_SUM_LIMIT,0.0416,4,Weak


IV survivors: ['EXT_SOURCE_1', 'BUR_DEBT_RATIO', 'BUR_PCT_ACTIVE', 'DAYS_EMPLOYED', 'BUR_OLDEST_DAYS', 'BUR_MOST_RECENT_DAYS', 'DAYS_LAST_PHONE_CHANGE', 'BUR_N_CLOSED', 'AMT_GOODS_PRICE', 'BUR_SUM_LIMIT', 'BUR_SUM_DEBT', 'FLAG_DOCUMENT_3', 'AMT_ANNUITY', 'BUR_LAST_UPDATE', 'AMT_CREDIT', 'FLOORSMAX_MODE', 'FLOORSMAX_AVG', 'FLOORSMAX_MEDI', 'TOTALAREA_MODE', 'BUR_SUM_CREDIT', 'BUR_MAX_ENDDATE', 'LIVINGAREA_AVG', 'LIVINGAREA_MODE', 'LIVINGAREA_MEDI', 'APARTMENTS_MODE', 'APARTMENTS_MEDI', 'APARTMENTS_AVG', 'OWN_CAR_AGE', 'YEARS_BEGINEXPLUATATION_MEDI', 'BUR_N_CONSUMER', 'YEARS_BEGINEXPLUATATION_AVG', 'BUR_MAX_AMT_MAX_OVERDUE', 'ELEVATORS_AVG', 'YEARS_BEGINEXPLUATATION_MODE', 'ELEVATORS_MEDI', 'DAYS_ID_PUBLISH', 'ELEVATORS_MODE', 'ENTRANCES_AVG']
Final candidates after correlation pruning: ['EXT_SOURCE_1', 'BUR_DEBT_RATIO', 'BUR_PCT_ACTIVE', 'DAYS_EMPLOYED', 'BUR_OLDEST_DAYS', 'BUR_MOST_RECENT_DAYS', 'DAYS_LAST_PHONE_CHANGE', 'BUR_N_CLOSED', 'AMT_GOODS_PRICE', 'BUR_SUM_LIMIT', 'BUR_SUM_DEBT

,Dropped,Kept,Absolute_Correlation
0,BUR_N_CONSUMER,BUR_N_CLOSED,0.927484
1,AMT_CREDIT,AMT_GOODS_PRICE,0.954959
2,FLOORSMAX_AVG,FLOORSMAX_MODE,0.987179
3,FLOORSMAX_MEDI,FLOORSMAX_MODE,0.989048
4,LIVINGAREA_AVG,TOTALAREA_MODE,0.923236
5,LIVINGAREA_MODE,TOTALAREA_MODE,0.901257
6,LIVINGAREA_MEDI,TOTALAREA_MODE,0.919962
7,APARTMENTS_MODE,TOTALAREA_MODE,0.862168
8,APARTMENTS_MEDI,TOTALAREA_MODE,0.884612
9,APARTMENTS_AVG,TOTALAREA_MODE,0.888746


In [10]:
# Coarse-class each surviving variable before fitting logistic regression.
coarse_models, coarse_tables, coarse_iv_rows = {}, {}, []
train_woe = pd.DataFrame(index=train_df.index)
test_woe = pd.DataFrame(index=test_df.index)
final_features = []
dropped_coarse_iv = []
coarse_business_reasons = {
    'EXT_SOURCE_1': 'External risk score; a stable non-linear relationship is plausible.',
    'BUR_DEBT_RATIO': 'Existing debt burden can have threshold and non-linear effects.',
    'BUR_PCT_ACTIVE': 'Active-credit proportion can have threshold effects.',
    'DAYS_EMPLOYED': 'Employment stability can have threshold effects.',
    'BUR_OLDEST_DAYS': 'Credit-history length can have threshold effects.',
    'DAYS_LAST_PHONE_CHANGE': 'Recent profile changes can indicate instability.',
    'AMT_CREDIT': 'Requested loan size can have affordability thresholds.',
    'DAYS_ID_PUBLISH': 'Profile-history age can have threshold effects.',
    'BUR_N_CLOSED': 'Closed-account history reflects prior credit-management behaviour.',
    'BUR_SUM_LIMIT': 'Total historical credit limits reflect available borrowing capacity.',
    'BUR_SUM_DEBT': 'Total historical debt measures existing repayment burden.',
    'BUR_LAST_UPDATE': 'Recency of bureau updates reflects current credit activity.',
    'AMT_GOODS_PRICE': 'Underlying purchase value reflects the affordability of the application.',
    'FLOORSMAX_MODE': 'Property-size information is a permissible affordability-related proxy.',
    'BUR_SUM_CREDIT': 'Total historical credit exposure reflects borrowing scale.',
    'TOTALAREA_MODE': 'Property size provides an affordability-related application signal.',
    'BUR_MOST_RECENT_DAYS': 'Recency of the latest bureau account reflects current credit history.',
    'BUR_MAX_AMT_MAX_OVERDUE': 'Maximum overdue amount directly reflects repayment difficulty.',
    'FLAG_DOCUMENT_3': 'Document-submission status is an application-process indicator.',
    'YEARS_BEGINEXPLUATATION_MEDI': 'Property age reflects collateral and affordability context.',
}


for feature in final_candidates:
    mapping, fine_table, coarse_table, woe_trend = coarse_class(
        fine_bins_train[feature],
        train_df['TARGET'],
        fine_models[feature],
        min_bin_pct=0.05,
        woe_tolerance=0.05,
    )
    coarse_iv = coarse_table['IV'].sum() if 'IV' in coarse_table else coarse_table['iv'].sum()

    fine_iv = fine_table['iv'].sum()
    fine_strength = iv_strength(fine_iv)
    coarse_strength = iv_strength(coarse_iv)
    iv_band_same = fine_strength == coarse_strength

    # Retain only monotonic or categorical variables. The non-monotonic
    # candidates were reviewed and did not show a clear, defensible business
    # shape such as a stable U-shape or threshold effect.
    if woe_trend == 'Stable non-monotonic':
        print(f'Dropped {feature}: non-monotonic WOE pattern is not sufficiently explainable.')
        continue

    # Apply the standard weak-IV cutoff after final coarse binning.
    if coarse_iv < 0.02:
        dropped_coarse_iv.append({'Variable': feature, 'Coarse_IV': coarse_iv})
        continue
    train_labels = fine_bins_train[feature].map(mapping)
    test_labels = fine_bins_test[feature].map(mapping)
    woe_mapping = woe_table_from_bins(train_labels, train_df['TARGET'])['woe'].to_dict()
    train_woe[feature] = train_labels.map(woe_mapping).fillna(0.0)
    test_woe[feature] = test_labels.map(woe_mapping).fillna(0.0)
    final_features.append(feature)
    coarse_models[feature] = mapping
    coarse_tables[feature] = woe_table_from_bins(train_labels, train_df['TARGET'])
    coarse_iv_rows.append({
        'Variable': feature,
        'Fine_IV': fine_iv,
        'Coarse_IV': coarse_iv,
        'Fine_IV_Strength': fine_strength,
        'Coarse_IV_Strength': coarse_strength,
        'IV_Band_Same': iv_band_same,
        'Business_Reason': coarse_business_reasons.get(feature, ''),
        'Coarse_Bins': len(coarse_tables[feature]),
        'Minimum_Bin_%': coarse_tables[feature]['count'].min() / len(train_df),
        'WOE_Trend': woe_trend,
        'IV_Strength': iv_strength(coarse_iv),
    })

if dropped_coarse_iv:
    print('Dropped after coarse IV check (IV < 0.02):')
    display(pd.DataFrame(dropped_coarse_iv).style.format({'Coarse_IV': '{:.4f}'}))
if not final_features:
    raise ValueError('No variables passed the final coarse IV check.')

coarse_iv_table = pd.DataFrame(coarse_iv_rows).sort_values('Coarse_IV', ascending=False).reset_index(drop=True)
display(coarse_iv_table.style.format({
    'Fine_IV': '{:.4f}',
    'Coarse_IV': '{:.4f}',
    'Minimum_Bin_%': '{:.2%}',
}))
print('IV strength-band changes require review:', int((~coarse_iv_table['IV_Band_Same']).sum()))

# The reviewed monotonic/categorical survivors provide the final scorecard
# variable set within the course's recommended 8-15 variable range.
print('Final WOE features:', final_features)
lr = LogisticRegression(solver='liblinear', max_iter=1000, random_state=453)


Dropped BUR_OLDEST_DAYS: non-monotonic WOE pattern is not sufficiently explainable.
Dropped AMT_GOODS_PRICE: non-monotonic WOE pattern is not sufficiently explainable.
Dropped BUR_SUM_LIMIT: non-monotonic WOE pattern is not sufficiently explainable.
Dropped BUR_SUM_DEBT: non-monotonic WOE pattern is not sufficiently explainable.
Dropped BUR_LAST_UPDATE: non-monotonic WOE pattern is not sufficiently explainable.
Dropped TOTALAREA_MODE: non-monotonic WOE pattern is not sufficiently explainable.
Dropped BUR_SUM_CREDIT: non-monotonic WOE pattern is not sufficiently explainable.
Dropped YEARS_BEGINEXPLUATATION_MEDI: non-monotonic WOE pattern is not sufficiently explainable.
Dropped after coarse IV check (IV < 0.02):


,Variable,Coarse_IV
0,BUR_DEBT_RATIO,0.0143
1,DAYS_LAST_PHONE_CHANGE,0.0198
2,AMT_ANNUITY,0.0070
3,BUR_MAX_ENDDATE,0.0142
4,OWN_CAR_AGE,0.0151
5,DAYS_ID_PUBLISH,0.0115
6,ENTRANCES_AVG,0.0162


,Variable,Fine_IV,Coarse_IV,Fine_IV_Strength,Coarse_IV_Strength,IV_Band_Same,Business_Reason,Coarse_Bins,Minimum_Bin_%,WOE_Trend,IV_Strength
0,BUR_PCT_ACTIVE,0.0832,0.0556,Weak,Weak,True,Active-credit proportion can have threshold effects.,4,10.51%,Monotonic after material merges,Weak
1,EXT_SOURCE_1,0.1686,0.0548,Medium,Weak,False,External risk score; a stable non-linear relationship is plausible.,3,19.63%,Monotonic after material merges,Weak
2,BUR_N_CLOSED,0.0457,0.0435,Weak,Weak,True,Closed-account history reflects prior credit-management behaviour.,3,13.99%,Monotonic after material merges,Weak
3,FLAG_DOCUMENT_3,0.0383,0.0383,Weak,Weak,True,Document-submission status is an application-process indicator.,2,29.83%,Categorical,Weak
4,FLOORSMAX_MODE,0.0321,0.0243,Weak,Weak,True,Property-size information is a permissible affordability-related proxy.,4,5.11%,Monotonic after material merges,Weak
5,DAYS_EMPLOYED,0.0818,0.0232,Weak,Weak,True,Employment stability can have threshold effects.,7,9.71%,Monotonic after material merges,Weak
6,BUR_MOST_RECENT_DAYS,0.0741,0.0226,Weak,Weak,True,Recency of the latest bureau account reflects current credit history.,5,8.36%,Monotonic after material merges,Weak
7,BUR_MAX_AMT_MAX_OVERDUE,0.0219,0.0219,Weak,Weak,True,Maximum overdue amount directly reflects repayment difficulty.,5,5.93%,Monotonic,Weak


IV strength-band changes require review: 1
Final WOE features: ['EXT_SOURCE_1', 'BUR_PCT_ACTIVE', 'DAYS_EMPLOYED', 'BUR_MOST_RECENT_DAYS', 'BUR_N_CLOSED', 'FLAG_DOCUMENT_3', 'FLOORSMAX_MODE', 'BUR_MAX_AMT_MAX_OVERDUE']


## 4. Fit logistic regression and calculate ROC and default probabilities for test and training data


In [11]:
# Fit logistic regression using the final coarse-bin WOE variables.
lr.fit(train_woe[final_features], train_df['TARGET'])
train_probability = lr.predict_proba(train_woe[final_features])[:, 1]
test_probability = lr.predict_proba(test_woe[final_features])[:, 1]
train_auc = roc_auc_score(train_df['TARGET'], train_probability)
test_auc = roc_auc_score(test_df['TARGET'], test_probability)
print(f'Training AUC: {train_auc:.4f}')
print(f'Test AUC: {test_auc:.4f}')
base_score, base_good_bad_odds, points_to_double_odds = 600, 19, 20


Training AUC: 0.6240
Test AUC: 0.6264


## 5. Function to Convert model probabilities into credit scores


In [12]:


# Course score scale: 600 score, 19:1 good-to-bad odds, 20 points per doubling.
base_score, base_good_bad_odds, points_to_double_odds = 600, 19, 20

# Higher scores represent lower predicted default risk.
factor = points_to_double_odds / np.log(2)
base_points = base_score + factor * (-lr.intercept_[0] - np.log(base_good_bad_odds))


def score_woe_frame(woe_frame):
    # Convert the model log-odds into the course credit-score scale.
    logit_bad = (
        lr.predict_log_proba(woe_frame)[:, 1]
        - lr.predict_log_proba(woe_frame)[:, 0]
    )
    return base_score + factor * (-logit_bad - np.log(base_good_bad_odds))

train_score = score_woe_frame(train_woe)
test_score = score_woe_frame(test_woe)
print('Score summary:')
print(pd.concat([
    pd.Series(train_score, name='train'),
    pd.Series(test_score, name='test')
], axis=1).describe().round(1))


Score summary:
         train     test
count  80515.0  34507.0
mean     582.5    582.5
std       13.4     13.3
min      552.2    552.2
25%      573.2    573.3
50%      581.7    581.7
75%      591.3    591.2
max      635.3    633.3


## 6. Build the points-based scorecard table


In [13]:
# Each row shows the points contributed by one coarse WOE bin.
scorecard_rows = []
for feature in final_features:
    coefficient = lr.coef_[0][list(final_features).index(feature)]
    for bin_name, row in coarse_tables[feature].iterrows():
        scorecard_rows.append({'Variable': feature, 'Bin': bin_name, 'Count': int(row['count']), 'Bad_Rate': row['bad'] / row['count'], 'WOE': row['woe'], 'IV': row['iv'], 'Points': -factor * coefficient * row['woe']})
scorecard_table = pd.DataFrame(scorecard_rows)
print('Base points:', round(base_points, 1))
display(scorecard_table.sort_values(['Variable', 'Bin']).style.format({'Bad_Rate': '{:.2%}', 'WOE': '{:.4f}', 'IV': '{:.4f}', 'Points': '{:.1f}'}))


Base points: 580.1


,Variable,Bin,Count,Bad_Rate,WOE,IV,Points
28,BUR_MAX_AMT_MAX_OVERDUE,Coarse bin 1,33426,8.07%,0.1780,0.0122,4.0
29,BUR_MAX_AMT_MAX_OVERDUE,Coarse bin 2,4775,11.27%,-0.1915,0.0024,-4.3
30,BUR_MAX_AMT_MAX_OVERDUE,Coarse bin 3,4775,11.29%,-0.1936,0.0024,-4.3
32,BUR_MAX_AMT_MAX_OVERDUE,Coarse bin 4,4775,11.37%,-0.2020,0.0026,-4.5
31,BUR_MAX_AMT_MAX_OVERDUE,Coarse bin 5,32764,10.15%,-0.0741,0.0023,-1.7
14,BUR_MOST_RECENT_DAYS,Coarse bin 1,6789,6.94%,0.3408,0.0085,2.5
15,BUR_MOST_RECENT_DAYS,Coarse bin 2,6730,8.01%,0.1858,0.0027,1.4
16,BUR_MOST_RECENT_DAYS,Coarse bin 3,40529,9.23%,0.0318,0.0005,0.2
18,BUR_MOST_RECENT_DAYS,Coarse bin 4,13579,10.19%,-0.0787,0.0011,-0.6
17,BUR_MOST_RECENT_DAYS,Coarse bin 5,12888,11.72%,-0.2359,0.0098,-1.8


## 7. Test an approval cutoff of 560
- Evaluating to see how cutoff would have impacted historical applicants in our datatset


In [14]:
# Applicants at or above the cutoff are approved; lower scores are classified as bad.
score_cutoff = 560
for title, actual, scores in [
    ('Training', train_df['TARGET'], train_score),
    ('Test', test_df['TARGET'], test_score)
]:
    approved = scores >= score_cutoff
    predicted_bad = (~approved).astype(int)
    print(
        f'{title}: approved {approved.sum():,} ({approved.mean():.1%}); '
        f'actual defaults among approved {actual[approved].sum():,}; '
        f'actual bad rate among approved {actual[approved].mean():.2%}'
    )
    print(classification_report(
        actual, predicted_bad, target_names=['Good payer', 'Defaulter'],
        digits=3, zero_division=0
    ))


Training: approved 77,941 (96.8%); actual defaults among approved 7,180; actual bad rate among approved 9.21%


              precision    recall  f1-score   support

  Good payer      0.908     0.971     0.938     72871
   Defaulter      0.180     0.061     0.091      7644

    accuracy                          0.885     80515
   macro avg      0.544     0.516     0.515     80515
weighted avg      0.839     0.885     0.858     80515

Test: approved 33,447 (96.9%); actual defaults among approved 3,085; actual bad rate among approved 9.22%
              precision    recall  f1-score   support

  Good payer      0.908     0.972     0.939     31231
   Defaulter      0.180     0.058     0.088      3276

    accuracy                          0.885     34507
   macro avg      0.544     0.515     0.513     34507
weighted avg      0.839     0.885     0.858     34507

